In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

# Create a small social network (Barabási–Albert scale-free)
G = nx.barabasi_albert_graph(100, 3)
np.random.seed(42)

# Edge activation probability (typical real-world: 0.01–0.1)
p = 0.05

def independent_cascade(G, seeds, p=0.05, steps=10): # 最多10輪循
    active = set(seeds)                    # 目前已經啟動（被影響）的節點集合
    newly_active = set(seeds)              # 本輪新啟動的節點（用來向外擴散）
    
    history = [len(active)]                # 記錄每一輪結束後累計啟動節點數，用於畫擴散曲線
    
    for t in range(steps):                 # 最多模擬 steps 輪
        next_newly = set()                 # 準備收集下一輪新啟動的節點
        
        # 遍歷本輪所有新啟動的節點，讓它們嘗試啟動自己的鄰居
        for node in newly_active:
            for neighbor in G.neighbors(node):   # 對當前節點的每個鄰居
                #  條件：鄰居還沒被啟動 且 以概率 p 成功啟動（獨立拋硬幣）
                if neighbor not in active and np.random.random() < p:
                    next_newly.add(neighbor)
        
        # 一輪結束，更新狀態
        newly_active = next_newly            # 下一輪要用這批新節點繼續向外擴散
        active.update(newly_active)         # 累計啟動集合加上新啟動的節點
        history.append(len(active))         # 記錄本輪結束後總啟動人數
        
        # 如果本輪沒有新節點被啟動 → 以後也不會再有擴散，直接提前結束
        if not newly_active:
            break
    
    return active, history                  # 返回最終啟動的節點集合 + 每一輪的累計啟動數量

# Seed a single random influencer 
seeds = [2]  # node 2 is a hub 先從種子2開始
final_active, sizes = independent_cascade(G, seeds, p=0.08)

print(f"Seed node {seeds} → reached {len(final_active)} nodes ({len(final_active)/100:.1%})")

# Plot cascade growth
plt.plot(sizes, marker='o')
plt.title("Information Cascade Size Over Time (IC Model)")
plt.xlabel("Time step")
plt.ylabel("Number of adopters")
plt.grid(True)
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random

def independent_cascade_model(G, seed_nodes, p=0.3):
    """
    Simulates the Independent Cascade Model.
    
    Args:
        G: NetworkX graph
        seed_nodes: List of initial active nodes
        p: Probability of activation (0.0 to 1.0)
    
    Returns:
        activated_nodes: Set of all nodes activated during the process
        activation_history: List of sets showing who activated at each step
    """
    # Initialize sets
    activated = set(seed_nodes)
    newly_activated = set(seed_nodes)
    history = [set(seed_nodes)]
    
    # Continue while there are nodes that just became active
    while newly_activated:
        next_round_activated = set()
        
        # For every node that became active in the previous round
        for u in newly_activated:
            # Check all neighbors of u
            for v in G.neighbors(u):
                # Only try to activate if v is not already active
                if v not in activated:
                    # Flip the coin: does u influence v?
                    if random.random() < p:
                        next_round_activated.add(v)
                        activated.add(v)
        
        # Update for next iteration
        newly_activated = next_round_activated
        if newly_activated:
            history.append(newly_activated)
            
    return activated, history

# --- Setup the Network ---
# Create a random graph (similar to Erdos-Renyi)
n_nodes = 50
p_edge = 0.1
G = nx.erdos_renyi_graph(n_nodes, p_edge, seed=42)

# Define seeds (e.g., 2 random starting nodes)
seeds = [5, 12] 
prop_prob = 0.4 # 40% chance of transmission

# --- Run Simulation ---
final_active, history = independent_cascade_model(G, seeds, p=prop_prob)

# --- Visualization ---
plt.figure(figsize=(12, 6))

# Layout for consistent visualization
pos = nx.spring_layout(G, seed=42)

# Plot 1: The Final State
plt.subplot(1, 2, 1)
nx.draw_networkx_nodes(G, pos, nodelist=list(final_active), node_color='red', node_size=300, label='Activated')
nx.draw_networkx_nodes(G, pos, nodelist=[n for n in G.nodes() if n not in final_active], node_color='lightgray', node_size=300, label='Inactive')
nx.draw_networkx_edges(G, pos, alpha=0.3)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title(f"IC Model Result\nSeeds: {seeds}, p={prop_prob}\nTotal Activated: {len(final_active)}")
plt.legend()
plt.axis('off')

# Plot 2: Activation Timeline (Step-by-Step)
plt.subplot(1, 2, 2)
nx.draw_networkx_nodes(G, pos, nodelist=seeds, node_color='green', node_size=400, label='Seeds (t=0)')
# Color code subsequent waves
colors = ['orange', 'purple', 'cyan', 'magenta']
current_idx = 0
for i, step_nodes in enumerate(history[1:], 1):
    color = colors[i % len(colors)]
    nx.draw_networkx_nodes(G, pos, nodelist=list(step_nodes), node_color=color, node_size=300, label=f'Step {i}')
    current_idx += 1

nx.draw_networkx_edges(G, pos, alpha=0.2)
plt.title("Activation Waves over Time")
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Simulation Complete.")
print(f"Initial Seeds: {seeds}")
print(f"Final Activated Count: {len(final_active)} out of {G.number_of_nodes()}")
print(f"Cascade Size Ratio: {len(final_active)/G.number_of_nodes():.2%}")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random

def linear_threshold_model(G, seed_nodes, weight_attr='weight'):
    """
    Simulates the Linear Threshold Model.
    
    Args:
        G: NetworkX graph (edges should ideally have weights, or default to 1/degree)
        seed_nodes: List of initial active nodes
        weight_attr: The edge attribute name for weights
    
    Returns:
        activated_nodes: Set of all activated nodes
        activation_history: List of sets showing who activated at each step
    """
    # 1. Assign Random Thresholds to all nodes (Uniform distribution 0 to 1)
    thresholds = {node: random.random() for node in G.nodes()}
    
    # 2. Normalize Edge Weights
    # Ensure sum of incoming weights for each node equals 1
    for node in G.nodes():
        in_edges = G.in_edges(node, data=True) if G.is_directed() else G.edges(node, data=True)
        total_weight = sum(data.get(weight_attr, 1.0) for _, _, data in in_edges)
        
        if total_weight > 0:
            for u, v, data in in_edges:
                # Normalize: new_weight = old_weight / total_incoming_weight
                # If undirected, u is the neighbor
                neighbor = u if v == node else v 
                # We update the graph edge attribute directly for simplicity
                if G.is_directed():
                    G[u][v][weight_attr] = data.get(weight_attr, 1.0) / total_weight
                else:
                    # For undirected, we just store the normalized contribution logic locally or update both
                    # Here we assume simple unweighted graph logic for visualization if weights missing
                    pass 
    
    # If graph was unweighted initially, let's create a temporary weighted view for calculation
    # Simplified approach for unweighted graphs: weight = 1 / degree
    if not nx.is_weighted(G):
        for node in G.nodes():
            degree = G.degree(node)
            if degree == 0: continue
            for neighbor in G.neighbors(node):
                # We will calculate influence dynamically below to avoid modifying original graph structure deeply
                pass

    # Initialize State
    activated = set(seed_nodes)
    newly_activated = set(seed_nodes)
    history = [set(seed_nodes)]
    
    # Simulation Loop
    while newly_activated:
        next_round_activated = set()
        
        # Check all INACTIVE nodes to see if they cross their threshold
        for node in G.nodes():
            if node in activated:
                continue
            
            # Calculate total influence from active neighbors
            influence_sum = 0.0
            neighbors = G.neighbors(node)
            
            # Determine normalization factor (sum of all incoming weights)
            # For unweighted graph: weight of each neighbor = 1 / degree
            degree = G.degree(node)
            if degree == 0:
                continue
                
            for neighbor in neighbors:
                if neighbor in activated:
                    # Contribution = 1 / degree (if unweighted)
                    # If weighted, use G[neighbor][node]['weight']
                    influence_sum += (1.0 / degree) 
            
            # Check Threshold Condition
            if influence_sum >= thresholds[node]:
                next_round_activated.add(node)
                activated.add(node)
        
        newly_activated = next_round_activated
        if newly_activated:
            history.append(newly_activated)
            
    return activated, history, thresholds

# --- Setup the Network ---
# Create a graph where community structure matters (e.g., Karate Club or Random)
G = nx.karate_club_graph() # A classic small social network

# Define seeds (Start with the instructor '0' and maybe one other)
seeds = [0] 

# --- Run Simulation ---
final_active, history, thresholds = linear_threshold_model(G, seeds)

# --- Visualization ---
plt.figure(figsize=(14, 6))
pos = nx.spring_layout(G, seed=42)

# Plot 1: Final State & Thresholds
plt.subplot(1, 2, 1)
node_colors = ['red' if node in final_active else 'lightgray' for node in G.nodes()]
node_sizes = [500 + (1 - thresholds[node]) * 1000 for node in G.nodes()] # Size inversely proportional to threshold (lower threshold = bigger/easier to activate)

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8, edgecolors='black')
nx.draw_networkx_edges(G, pos, alpha=0.3)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')

plt.title(f"LT Model Result\nSeeds: {seeds}\nActivated: {len(final_active)}/{G.number_of_nodes()}")
plt.text(0.5, -0.1, "Node Size = Susceptibility (Inverse of Threshold)\nRed = Activated, Gray = Resistant", 
         transform=plt.gca().transAxes, ha='center', fontsize=10, bbox=dict(facecolor='white', alpha=0.5))
plt.axis('off')

# Plot 2: Activation Waves
plt.subplot(1, 2, 2)
nx.draw_networkx_nodes(G, pos, nodelist=seeds, node_color='green', node_size=600, label='Seeds (t=0)', edgecolors='black')
colors = ['orange', 'purple', 'cyan', 'magenta']

for i, step_nodes in enumerate(history[1:], 1):
    color = colors[i % len(colors)]
    nx.draw_networkx_nodes(G, pos, nodelist=list(step_nodes), node_color=color, node_size=400, label=f'Step {i}', edgecolors='black')

nx.draw_networkx_edges(G, pos, alpha=0.2)
plt.title("Activation Waves (Cumulative Pressure)")
plt.legend(loc='best')
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Simulation Complete.")
print(f"Threshold Stats: Min={min(thresholds.values()):.2f}, Max={max(thresholds.values()):.2f}")
print(f"Cascade Size: {len(final_active)}")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random

def triggering_model(G, seed_nodes, p_attr='prob'):
    """
    Simulates the Triggering Model.
    
    Args:
        G: NetworkX directed graph. Edges should have a 'prob' attribute (default 0.5).
        seed_nodes: List of initial active nodes.
        p_attr: Edge attribute name for triggering probability.
    
    Returns:
        activated_nodes: Set of all activated nodes.
        triggering_sets: Dictionary mapping each node to its selected triggering set.
        activation_history: List of sets showing activation steps.
    """
    # --- Phase 1: Construct Triggering Sets ---
    # Every node v selects a subset of its predecessors based on edge probabilities
    triggering_sets = {node: set() for node in G.nodes()}
    
    for v in G.nodes():
        # Look at all incoming edges (predecessors)
        for u in G.predecessors(v):
            prob = G[u][v].get(p_attr, 0.5) # Default 0.5 if not specified
            if random.random() < prob:
                triggering_sets[v].add(u)
    
    # --- Phase 2: Run the Cascade ---
    activated = set(seed_nodes)
    newly_activated = set(seed_nodes)
    history = [set(seed_nodes)]
    
    while newly_activated:
        next_round_activated = set()
        
        # Check all inactive nodes
        for v in G.nodes():
            if v in activated:
                continue
            
            # Check if ANY member of v's triggering set is now active
            # Intersection of Triggering Set and Currently Active Nodes
            if triggering_sets[v] & activated: 
                next_round_activated.add(v)
                activated.add(v)
        
        newly_activated = next_round_activated
        if newly_activated:
            history.append(newly_activated)
            
    return activated, triggering_sets, history

# --- Setup the Network ---
# Create a Directed Graph (Influence is often directional)
G = nx.karate_club_graph() # Using Karate club but treating as directed for demo
G = G.to_directed()

# Assign random probabilities to edges (simulating different trust levels)
for u, v in G.edges():
    G[u][v]['prob'] = random.uniform(0.1, 0.6)

# Define Seeds
seeds = [0] # The instructor

# --- Run Simulation ---
final_active, t_sets, history = triggering_model(G, seeds)

# --- Visualization ---
plt.figure(figsize=(14, 6))
pos = nx.spring_layout(G, seed=42)

# Plot 1: The Triggering Structure (Who listens to whom?)
plt.subplot(1, 2, 1)
# Draw all possible edges faintly
nx.draw_networkx_edges(G, pos, alpha=0.1, edge_color='gray', style='dashed')

# Draw ONLY the edges that ended up in a Triggering Set (The "Real" Influence Paths)
trigger_edges = []
for v, triggers in t_sets.items():
    for u in triggers:
        trigger_edges.append((u, v))

nx.draw_networkx_edges(G, pos, edgelist=trigger_edges, width=2.0, edge_color='red', alpha=0.8, arrowstyle='-|>', arrowsize=15)
nx.draw_networkx_nodes(G, pos, nodelist=list(final_active), node_color='red', node_size=400, label='Activated')
nx.draw_networkx_nodes(G, pos, nodelist=[n for n in G.nodes() if n not in final_active], node_color='lightgray', node_size=400, label='Inactive')
nx.draw_networkx_labels(G, pos, font_size=8)

plt.title(f"Triggering Model: Realized Influence Paths\nRed Arrows = Active Triggers")
plt.legend()
plt.axis('off')

# Plot 2: Activation Waves
plt.subplot(1, 2, 2)
nx.draw_networkx_nodes(G, pos, nodelist=seeds, node_color='green', node_size=600, label='Seeds', edgecolors='black')
colors = ['orange', 'purple', 'cyan', 'magenta']

for i, step_nodes in enumerate(history[1:], 1):
    color = colors[i % len(colors)]
    nx.draw_networkx_nodes(G, pos, nodelist=list(step_nodes), node_color=color, node_size=400, label=f'Step {i}', edgecolors='black')

# Show the realized trigger edges again for context
nx.draw_networkx_edges(G, pos, edgelist=trigger_edges, width=1.0, edge_color='red', alpha=0.5, arrowstyle='-|>', arrowsize=10)
nx.draw_networkx_edges(G, pos, edgelist=[e for e in G.edges() if e not in trigger_edges], width=0.5, edge_color='gray', alpha=0.1, style='dashed')

plt.title("Cascade Propagation via Triggering Sets")
plt.legend(loc='best')
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Simulation Complete.")
print(f"Total Possible Edges: {G.number_of_edges()}")
print(f"Active Triggering Links: {len(trigger_edges)}")
print(f"Cascade Size: {len(final_active)}")

In [ ]:
import networkx as nx
import random
import matplotlib.pyplot as plt
from itertools import combinations

# --- Helper: Run IC Simulation (from previous step) ---
def independent_cascade_spread(G, seed_nodes, p=0.1, iterations=100):
    """
    Estimates the expected spread sigma(S) by averaging multiple simulations.
    """
    total_spread = 0
    for _ in range(iterations):
        activated = set(seed_nodes)
        newly_activated = set(seed_nodes)
        
        while newly_activated:
            next_round = set()
            for u in newly_activated:
                for v in G.neighbors(u):
                    if v not in activated:
                        if random.random() < p:
                            next_round.add(v)
                            activated.add(v)
            newly_activated = next_round
        
        total_spread += len(activated)
    
    return total_spread / iterations

# --- The Greedy Algorithm for Influence Maximization ---
def greedy_influence_maximization(G, k, p=0.1, simulations=50):
    """
    Finds the best k seed nodes using the Greedy approach.
    
    Args:
        G: NetworkX graph
        k: Number of seeds to select
        p: Propagation probability
        simulations: Number of MC simulations per evaluation
    
    Returns:
        selected_seeds: List of k best nodes
        spread_history: List of expected spread after each selection
    """
    selected_seeds = []
    spread_history = []
    current_spread = 0
    
    print(f"Starting Greedy Search for k={k} seeds...")
    
    for i in range(k):
        best_node = None
        max_marginal_gain = -1
        
        # Evaluate every node NOT yet selected
        # Note: This is the expensive part O(N * k * Simulations)
        candidates = [n for n in G.nodes() if n not in selected_seeds]
        
        print(f"  Selecting seed #{i+1}... evaluating {len(candidates)} candidates.")
        
        for candidate in candidates:
            # Calculate spread if we add this candidate to current set
            temp_seeds = selected_seeds + [candidate]
            expected_spread = independent_cascade_spread(G, temp_seeds, p=p, iterations=simulations)
            
            # Marginal Gain = New Spread - Current Spread
            marginal_gain = expected_spread - current_spread
            
            if marginal_gain > max_marginal_gain:
                max_marginal_gain = marginal_gain
                best_node = candidate
        
        # Add the best node found
        selected_seeds.append(best_node)
        current_spread += max_marginal_gain
        spread_history.append(current_spread)
        print(f"  -> Selected Node: {best_node} (Marginal Gain: {max_marginal_gain:.2f}, Total Expected Spread: {current_spread:.2f})")
    
    return selected_seeds, spread_history

# --- Setup Network ---
# Using a larger graph to make IM meaningful
G = nx.barabasi_albert_graph(n=200, m=3, seed=42) # Scale-free network
p_prob = 0.05 # Low probability to make selection critical
k_seeds = 5

# --- Run Optimization ---
# WARNING: Greedy IM is slow for large graphs. 
# For n=200, it's fine. For n=100,000, you need approximations (like CELF or TIM).
best_seeds, history = greedy_influence_maximization(G, k=k_seeds, p=p_prob, simulations=20)

# --- Visualization ---
plt.figure(figsize=(12, 5))

# Plot 1: The Selected Seeds on the Network
plt.subplot(1, 2, 1)
pos = nx.spring_layout(G, seed=42)
node_colors = ['red' if node in best_seeds else 'lightblue' for node in G.nodes()]
node_sizes = [800 if node in best_seeds else 100 for node in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8)
nx.draw_networkx_edges(G, pos, alpha=0.1)
nx.draw_networkx_labels(G, pos, font_size=8, labels={n: n for n in best_seeds}) # Only label seeds

plt.title(f"Optimal Seeds (k={k_seeds})\nNodes: {best_seeds}")
plt.axis('off')

# Plot 2: Diminishing Returns (Submodularity)
plt.subplot(1, 2, 2)
plt.plot(range(1, k_seeds + 1), history, marker='o', linestyle='-', color='green', linewidth=2)
plt.xlabel("Number of Seeds (k)")
plt.ylabel("Expected Influence Spread")
plt.title("Diminishing Returns (Submodularity)")
plt.grid(True, alpha=0.3)
plt.xticks(range(1, k_seeds + 1))

# Annotate the curve
for i, v in enumerate(history):
    plt.text(i+1, v, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nOptimization Complete.")
print(f"Best Seeds: {best_seeds}")
print(f"Final Expected Spread: {history[-1]:.2f} nodes")

In [ ]:
import networkx as nx
import random
import matplotlib.pyplot as plt
import copy

# --- 1. Simulation: Independent Cascade with Removed Nodes ---
def simulate_spread_with_removal(G, seed_nodes, p=0.3, removed_nodes=None):
    """
    Simulates IC spread on a graph where specific nodes are removed/immunized.
    Removed nodes cannot be activated and cannot transmit influence.
    """
    if removed_nodes is None:
        removed_nodes = set()
    
    # Create a view of the graph without the removed nodes
    # We don't modify the original G, just ignore removed nodes during traversal
    active_seeds = [n for n in seed_nodes if n not in removed_nodes]
    
    activated = set(active_seeds)
    newly_activated = set(active_seeds)
    
    while newly_activated:
        next_round = set()
        for u in newly_activated:
            for v in G.neighbors(u):
                # Skip if v is already active, is a seed, or is REMOVED
                if v in activated or v in removed_nodes:
                    continue
                
                # Try to infect
                if random.random() < p:
                    next_round.add(v)
                    activated.add(v)
        
        newly_activated = next_round
    
    return len(activated)

# --- 2. Optimization: Greedy Node Removal ---
def greedy_node_removal(G, bad_seeds, k, p=0.3, simulations=50):
    """
    Finds the best k nodes to REMOVE to minimize the spread of bad_seeds.
    """
    removed_nodes = set()
    history = []
    
    # Calculate baseline spread (no removal)
    baseline_spread = 0
    for _ in range(simulations):
        baseline_spread += simulate_spread_with_removal(G, bad_seeds, p=p, removed_nodes=set())
    baseline_spread /= simulations
    
    print(f"Baseline Spread (No Removal): {baseline_spread:.2f} nodes")
    print(f"Starting Greedy Removal Search for k={k} nodes...")
    
    for i in range(k):
        best_node = None
        min_spread = float('inf')
        
        # Candidates: All nodes except seeds and already removed ones
        # Note: Removing a seed itself is often the most effective strategy!
        candidates = [n for n in G.nodes() if n not in removed_nodes]
        
        print(f"  Selecting node #{i+1} to remove... evaluating {len(candidates)} candidates.")
        
        for candidate in candidates:
            current_removed = removed_nodes | {candidate}
            
            total_spread = 0
            for _ in range(simulations):
                spread = simulate_spread_with_removal(G, bad_seeds, p=p, removed_nodes=current_removed)
                total_spread += spread
            
            avg_spread = total_spread / simulations
            
            if avg_spread < min_spread:
                min_spread = avg_spread
                best_node = candidate
        
        if best_node is not None:
            removed_nodes.add(best_node)
            history.append(min_spread)
            gain = (history[-2] if i > 0 else baseline_spread) - min_spread
            print(f"  -> Removed Node: {best_node} (New Expected Spread: {min_spread:.2f}, Reduction: {gain:.2f})")
    
    return list(removed_nodes), history, baseline_spread

# --- 3. Setup Scenario ---
# Create a scale-free network (prone to hubs)
G = nx.barabasi_albert_graph(n=150, m=3, seed=42)

# Define "Bad Seeds" (The source of the panic)
# Let's pick a high-degree node that is NOT the absolute highest, to make it interesting
degrees = dict(G.degree())
sorted_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)
bad_seeds = [sorted_nodes[2][0]] # Pick the 3rd largest hub as the panic source

p_prob = 0.4
k_removals = 3

# --- 4. Run Optimization ---
optimal_removals, reduction_history, baseline = greedy_node_removal(
    G, bad_seeds, k=k_removals, p=p_prob, simulations=30
)

# --- 5. Visualization ---
plt.figure(figsize=(14, 6))
pos = nx.spring_layout(G, seed=42)

# Determine final states for coloring (simulate once with optimal removals)
final_active_count = simulate_spread_with_removal(G, bad_seeds, p=p_prob, removed_nodes=set(optimal_removals))
# To visualize who got infected, we need a slightly modified sim that returns sets
def get_infected_set(G, seeds, p, removed):
    activated = set([n for n in seeds if n not in removed])
    newly = set(activated)
    while newly:
        nxt = set()
        for u in newly:
            for v in G.neighbors(u):
                if v not in activated and v not in removed:
                    if random.random() < p:
                        nxt.add(v); activated.add(v)
        newly = nxt
    return activated

infected_nodes = get_infected_set(G, bad_seeds, p_prob, set(optimal_removals))

node_colors = []
node_sizes = []
for node in G.nodes():
    if node in bad_seeds:
        node_colors.append('red')       # Source of Panic
        node_sizes.append(600)
    elif node in optimal_removals:
        node_colors.append('black')     # Removed/Immunized (The "Wall")
        node_sizes.append(500)
    elif node in infected_nodes:
        node_colors.append('orange')    # Infected
        node_sizes.append(200)
    else:
        node_colors.append('lightgreen')# Saved/Immune
        node_sizes.append(100)

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, edgecolors='gray', linewidths=0.5)
nx.draw_networkx_edges(G, pos, alpha=0.1)

# Draw edges connected to removed nodes differently to show "cuts"? 
# For simplicity, we just highlight the nodes.

legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=15, label='Panic Source'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=15, label='Removed (Immunized)'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=10, label='Infected'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightgreen', markersize=10, label='Saved')
]
plt.legend(handles=legend_elements, loc='upper left')
plt.title(f"Node Removal Strategy\nRemoved: {optimal_removals}\nSpread Reduced from {baseline:.1f} to {reduction_history[-1]:.1f}")
plt.axis('off')
plt.tight_layout()
plt.show()

# Plot Reduction Curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, k_removals + 1), reduction_history, marker='s', color='red', linestyle='--')
plt.axhline(y=baseline, color='gray', linestyle='-', label='Baseline (No Removal)')
plt.xlabel("Number of Nodes Removed (k)")
plt.ylabel("Expected Infection Count")
plt.title("Effectiveness of Node Removal (Immunization)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nStrategy Complete.")
print(f"Optimal Nodes to Remove: {optimal_removals}")
print(f"Total Spread Reduction: {baseline - reduction_history[-1]:.2f} nodes")

In [ ]:
import networkx as nx
import random
import matplotlib.pyplot as plt

def competing_cascade_simulation(G, bad_seeds, good_seeds, p_bad=0.3, p_good=0.3):
    """
    Simulates two competing cascades (Bad vs Good).
    Rule: First come, first served. If 'Good' reaches a node first (or at same time), it is saved.
    """
    # States: 0=Inactive, 1=Bad, 2=Good (Saved)
    state = {node: 0 for node in G.nodes()}
    
    # Initialize seeds
    for n in bad_seeds: state[n] = 1
    for n in good_seeds: 
        if state[n] == 0: state[n] = 2 # If not already bad, become good
    
    active_bad = set(bad_seeds)
    active_good = set([n for n in good_seeds if state[n] == 2])
    
    while active_bad or active_good:
        next_bad = set()
        next_good = set()
        
        # Process Bad Spread
        for u in active_bad:
            for v in G.neighbors(u):
                if state[v] == 0: # Only try to infect inactive
                    if random.random() < p_bad:
                        # Tentatively mark, but check collision later if needed
                        # Simplified: Order of processing matters slightly, 
                        # but usually we assume simultaneous update or priority to Good
                        next_bad.add(v)
        
        # Process Good Spread (Defense)
        for u in active_good:
            for v in G.neighbors(u):
                if state[v] == 0: # Only try to save inactive
                    if random.random() < p_good:
                        next_good.add(v)
                elif state[v] == 1 and v in next_bad:
                    # Collision: Bad tried to infect, Good also reached. 
                    # Let's give priority to Good (Defense wins ties)
                    next_bad.discard(v)
                    next_good.add(v)

        # Update State
        for v in next_good:
            if state[v] == 0: state[v] = 2
        for v in next_bad:
            if state[v] == 0: state[v] = 1 # Only infect if not saved by good
            
        active_bad = {v for v in next_bad if state[v] == 1}
        active_good = {v for v in next_good if state[v] == 2}
        
        if not active_bad and not active_good:
            break
            
    # Count outcomes
    bad_count = sum(1 for s in state.values() if s == 1)
    saved_count = sum(1 for s in state.values() if s == 2)
    return bad_count, saved_count, state

def greedy_blocking_strategy(G, bad_seeds, k, p_bad=0.3, p_good=0.3, simulations=20):
    """
    Finds the best k 'good' seeds to minimize the spread of 'bad' seeds.
    """
    selected_defenders = []
    
    # Calculate baseline bad spread without any defense
    baseline_bad, _, _ = competing_cascade_simulation(G, bad_seeds, [], p_bad, p_good)
    # Actually, we want to maximize SAVED nodes. 
    # Baseline saved is just the good seeds themselves (if any) or 0 if we start with none.
    
    print(f"Starting Defensive Search for k={k} defenders...")
    
    for i in range(k):
        best_node = None
        max_saved_gain = -1
        
        candidates = [n for n in G.nodes() if n not in selected_defenders and n not in bad_seeds]
        print(f"  Selecting defender #{i+1}... evaluating {len(candidates)} candidates.")
        
        for candidate in candidates:
            current_seeds = selected_defenders + [candidate]
            
            total_saved = 0
            for _ in range(simulations):
                _, saved, _ = competing_cascade_simulation(G, bad_seeds, current_seeds, p_bad, p_good)
                total_saved += saved
            
            avg_saved = total_saved / simulations
            
            # Calculate gain over previous iteration
            # (Simplified: comparing to current best configuration)
            # Ideally compare marginal gain: Saved(with candidate) - Saved(without candidate)
            # Here we just track the absolute best avg_saved found so far for this step
            if avg_saved > max_saved_gain:
                max_saved_gain = avg_saved
                best_node = candidate
        
        if best_node is not None:
            selected_defenders.append(best_node)
            print(f"  -> Selected Defender: {best_node} (Expected Saved: {max_saved_gain:.2f})")
            
    return selected_defenders

# --- Setup Scenario ---
G = nx.barabasi_albert_graph(n=150, m=3, seed=42)

# 1. The Threat: A few hubs start a panic sell (Bad Seeds)
# Let's pick high degree nodes as the threat
degrees = dict(G.degree())
sorted_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)
bad_seeds = [sorted_nodes[0][0], sorted_nodes[1][0]] # Top 2 hubs are panicked

# 2. The Defense Budget
k_defenders = 3

# --- Run Optimization ---
best_defenders = greedy_blocking_strategy(G, bad_seeds, k=k_defenders, p_bad=0.4, p_good=0.4, simulations=15)

# --- Final Visualization ---
plt.figure(figsize=(14, 6))
pos = nx.spring_layout(G, seed=42)

# Run one final sim to get states for coloring
_, _, final_state = competing_cascade_simulation(G, bad_seeds, best_defenders, p_bad=0.4, p_good=0.4)

node_colors = []
for node in G.nodes():
    if node in bad_seeds:
        node_colors.append('red')       # Source of Panic
    elif node in best_defenders:
        node_colors.append('green')     # Defenders
    elif final_state[node] == 1:
        node_colors.append('orange')    # Infected (Failed to save)
    elif final_state[node] == 2:
        node_colors.append('lightgreen')# Saved
    else:
        node_colors.append('gray')      # Never reached

node_sizes = [100 if c == 'gray' else 300 for c in node_colors]
# Highlight seeds/defenders
for i, node in enumerate(G.nodes()):
    if node in bad_seeds: node_sizes[i] = 600
    if node in best_defenders: node_sizes[i] = 500

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8, edgecolors='black')
nx.draw_networkx_edges(G, pos, alpha=0.1)

legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=15, label='Panic Source (Bad)'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=15, label='Strategic Defender'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=10, label='Infected (Lost)'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightgreen', markersize=10, label='Saved (Protected)')
]
plt.legend(handles=legend_elements, loc='upper left')
plt.title(f"Cascade Blocking Result\nBad Seeds: {bad_seeds} | Defenders: {best_defenders}")
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"\nDefense Complete.")
print(f"Panic Sources: {bad_seeds}")
print(f"Optimal Defenders: {best_defenders}")